In [2]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
model="gemini-2.5-flash",
temperature=0
)

In [4]:
import sys
sys.path.append("..")

In [5]:
from src.tools import read_calendar, get_customer_profile
tools = {
"read_calendar": read_calendar,
"get_customer_profile": get_customer_profile
}

In [6]:
def triage_node(state):
    email = state["email"]
    prompt = f"""
    Classify this email into one of:
    ignore
    notify_human
    respond
    Email:
    {email}
    Return only the label.
    """
    label = llm.invoke(prompt).content.strip().lower()
    return {**state, "triage": label}

In [7]:
def react_agent(state):
    email = state["email"]

    prompt = f"""
You are an email assistant.
You can use tools if needed.
Tools:
read_calendar
get_customer_profile
Email:
{email}
If you need a tool, write TOOL:<toolname>
Otherwise give reply.
"""

    response = llm.invoke(prompt).content

    if "TOOL:" in response:
        tool_name = response.replace("TOOL:", "").strip()
        tool_result = tools[tool_name]()
        return {**state, "response": tool_result}

    return {**state, "response": response}


In [8]:
from langgraph.graph import StateGraph
graph = StateGraph(dict)
graph.add_node("triage", triage_node)
graph.add_node("react", react_agent)
def route(state):
 if state["triage"] == "respond":
    return "react"
 else:
    return "end"
graph.add_conditional_edges("triage", route)
graph.set_entry_point("triage")
app = graph.compile()

In [1]:
import pandas as pd
emails = pd.read_csv("../data/sample_emails_with_triage_200.csv")

In [9]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

print("Tracing:", os.getenv("LANGCHAIN_TRACING_V2"))
print("Project:", os.getenv("LANGCHAIN_PROJECT"))
print("Key:", os.getenv("LANGCHAIN_API_KEY")[:8])


Tracing: true
Project: Infosys-Milestone-1
Key: lsv2_pt_


In [10]:
from langsmith import Client
client = Client()
client.list_projects()

<generator object Client.list_projects at 0x0000022E860F2110>

In [11]:
list(client.list_projects())

[TracerSessionResult(id=UUID('670d316b-20e3-4a40-8e17-a345c187337e'), start_time=datetime.datetime(2026, 1, 13, 13, 47, 38, 270740, tzinfo=datetime.timezone.utc), end_time=None, description=None, name='Infosys-Milestone-1', extra=None, tenant_id=UUID('15ab1968-d5ff-43dd-93dc-9e106b0849a8'), reference_dataset_id=None, run_count=None, latency_p50=None, latency_p99=None, total_tokens=None, prompt_tokens=None, completion_tokens=None, last_run_start_time=None, feedback_stats=None, session_feedback_stats=None, run_facets=None, total_cost=None, prompt_cost=None, completion_cost=None, first_token_p50=None, first_token_p99=None, error_rate=None)]

In [13]:
results = []

for _, row in emails.head(5).iterrows():
    email = row["body"]
    output = app.invoke({"email": email})

    results.append({
        "email": email,
        "triage": output["triage"],
        "response": output.get("response", "")
    })


Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.


In [20]:
pd.DataFrame(results).to_csv("../data/milestone1_output.csv", index=False)

### doubt

In [25]:
import pandas as pd
gold = pd.read_csv("../data/golden_labels.csv")
pred = pd.read_csv("../data/milestone1_output.csv")


merged = gold.merge(
pred,
on="email",
how="inner",
suffixes=("_gold", "_pred")
)

accuracy = (merged["expected"] ==
merged["triage"]).mean()
accuracy

nan

In [ ]:
"""Open:
https://smith.langchain.com
Select project:
Infosys-Milestone-1
You must see:
• Triage decisions
• ReAct reasoning
• Tool calls"""

In [ ]:
"""What You Must Submit
Each intern must push:
data/milestone1_output.csv
data/golden_labels.csv
notebook/milestone1_final.ipynb
If any of you gets API quota error, you must wait and retry — Gemini is required as per project
document."""

In [ ]:
"""merged = gold.merge(
pred,
on="email",
how="inner",
suffixes=("_gold", "_pred")
)

accuracy = (merged["expected"] ==
merged["triage"]).mean()
accuracy"""